In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import statsmodels.api as sm

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'WenQuanYi Micro Hei']
plt.rcParams['axes.unicode_minus'] = False


In [6]:

def build_dsp_factor(original, filtered, lookback=60):
    """
    构建DSP增强因子
    """
    common_idx = original.index.intersection(filtered.index)
    orig_aligned = original.loc[common_idx]
    filt_aligned = filtered.loc[common_idx]
    
    filtered_momentum = filt_aligned.pct_change(periods=lookback)
    residual = orig_aligned.values - filt_aligned.values
    residual_series = pd.Series(residual, index=common_idx)
    noise_intensity = residual_series.rolling(window=lookback).std()
    dsp_factor = filtered_momentum / (noise_intensity + 1e-8)
    original_momentum = orig_aligned.pct_change(periods=lookback)
    
    return {
        'dsp_factor': dsp_factor,
        'filtered_momentum': filtered_momentum,
        'original_momentum': original_momentum,
        'noise_intensity': noise_intensity
    }

# ---------------------------------------------------------------------
# 1. 读取数据
# ---------------------------------------------------------------------
df_all = pd.read_csv(r"..\数据\全股票滤波结果总表.csv")
df_all['trade_date'] = pd.to_datetime(df_all['trade_date'])

# ---------------------------------------------------------------------
# 2. 计算原始 DSP 因子
# ---------------------------------------------------------------------
factors_dict = {}
factor_rows = []

for code, df_stock in df_all.groupby('code'):
    original = df_stock['close_price']
    filtered = df_stock['filtered_price']
    
    factors = build_dsp_factor(original, filtered, lookback=60)
    factors_dict[code] = factors
    
    temp = pd.DataFrame({
        'trade_date': df_stock['trade_date'], 
        'code': code,
        'close_price': original,
        'filtered_price': filtered,
        'dsp_raw': factors['dsp_factor'],  # 改名：原始因子
        'filtered_momentum': factors['filtered_momentum'],
        'original_momentum': factors['original_momentum'],
        'noise_intensity': factors['noise_intensity'],
        'industry': df_stock['industry'],  # 行业
        'market_cap': df_stock['market_cap']# 市值
    })
    factor_rows.append(temp)

df_factor_all = pd.concat(factor_rows)

# ---------------------------------------------------------------------
# =====================因子中性化 =====================
# ---------------------------------------------------------------------
def neutralize_factor_daily(df):
    """ 每日截面：行业 + 市值 中性化 """
    def _neutralize(group):
        log_cap = np.log(group['market_cap'])
        ind_dummies = pd.get_dummies(group['industry'], drop_first=True)
        X = pd.concat([ind_dummies, log_cap.rename('log_cap')], axis=1)
        X = sm.add_constant(X)
        y = group['dsp_raw']

        try:
            res = sm.OLS(y, X).fit().resid
        except:
            res = y

        group['dsp_factor'] = res  # 最终中性化因子
        return group

    return df.groupby('trade_date', group_keys=False).apply(_neutralize)

# 执行中性化
df_factor_all = neutralize_factor_daily(df_factor_all)

# ---------------------------------------------------------------------
# 3. 保存最终结果
# ---------------------------------------------------------------------
df_final = df_factor_all[[
    'trade_date', 'code', 'close_price', 'filtered_price',
    'dsp_factor', 'filtered_momentum', 'original_momentum', 'noise_intensity'
]].sort_values(['code', 'trade_date'])

save_path = r"..\数据\DSP因子总表.csv"
df_final.to_csv(save_path, index=False, encoding='utf-8-sig')
print("\n✅ 完成！DSP因子已中性化 → 全市场可直接回测")

C:\Users\wangwei\AppData\Local\Temp\ipykernel_1140\1595957323.py:78: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('trade_date', group_keys=False).apply(_neutralize)



✅ 完成！DSP因子已中性化 → 全市场可直接回测
